# 16: Live demo

A web page where anyone can upload an image and see:
1. **Verdict** from our best detector (M2 trained on OpenFake, E7)
2. **All detectors' scores**, the 2023-trained ones (M1-M4) and the 2025-trained ones (E7)
3. **Closest GenImage source** from the seven-class attribution model (E4)
4. **Fingerprint view**: the image's noise residual, its frequency spectrum and its NPR residual

Run all, then open the public link printed at the bottom (valid for 72 hours while this notebook keeps running). GPU runtime recommended.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gradio pillow-heif

import json
from pathlib import Path

import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pillow_heif
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import LogisticRegression
from torchvision.models import resnet50
from transformers import CLIPVisionModelWithProjection

pillow_heif.register_heif_opener()   # lets iPhone HEIC photos be uploaded directly
ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
SIZE = 224
CLIP_NAME = "openai/clip-vit-large-patch14"
SOURCES = ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"]

## Load every trained model

The two logistic-regression classifiers on CLIP features are re-fitted from the saved features with the C chosen in notebooks 06 and 15 (the fit is deterministic).

In [ ]:
class NPRNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet50(weights=None)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
            base.bn1, base.relu, base.maxpool,
        )
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        rebuilt = F.interpolate(F.interpolate(x, scale_factor=0.5, mode="nearest"), scale_factor=2, mode="nearest")
        npr = x - rebuilt
        x = self.stem(npr * 2.0 / 3.0)
        x = self.layer2(self.layer1(x))
        return self.fc(torch.flatten(self.pool(x), 1))


def load_resnet(name, outputs=1):
    net = resnet50(weights=None)
    net.fc = nn.Linear(net.fc.in_features, outputs)
    net.load_state_dict(torch.load(ROOT / "checkpoints" / f"{name}.pt", map_location=device))
    return net.to(device).eval()


m1_old = load_resnet("m1_resnet50")
m1_new = load_resnet("e7_m1_openfake")
attributor = load_resnet("e4_a2_resnet50", outputs=len(SOURCES))
m3_old = NPRNet()
m3_old.load_state_dict(torch.load(ROOT / "checkpoints" / "m3_npr.pt", map_location=device))
m3_old = m3_old.to(device).eval()

dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size

old_feats = np.load(ROOT / "features" / f"{CLIP_NAME.split('/')[-1]}_train.npy")
old_y = (pd.read_csv(ROOT / "processed" / "train_meta.csv")["label"] == "fake").astype(int).to_numpy()
m2_old = LogisticRegression(C=json.load(open(RESULTS / "m2_clip.json"))["best_C"], max_iter=5000).fit(old_feats, old_y)

new_meta = pd.read_csv(ROOT / "processed_openfake_train" / "meta.csv")
new_feats = np.load(ROOT / "processed_openfake_train" / "clip_features.npy")
train = (new_meta["split"] == "train").to_numpy()
new_y = (new_meta["label"] == "fake").astype(int).to_numpy()
new_C = json.load(open(RESULTS / "e7_m1_openfake.json"))["best_C"]
m2_new = LogisticRegression(C=new_C, max_iter=5000).fit(new_feats[train], new_y[train])
print("All models loaded")

## Analysis of one image

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def fingerprint_figure(crop):
    gray = crop.mean(axis=2)
    residual = gray - gaussian_filter(gray, 1)
    spectrum = np.log(np.abs(np.fft.fftshift(np.fft.fft2(residual))) + 1e-3)
    x = crop.astype(np.float32)
    npr = x - np.repeat(np.repeat(x[::2, ::2], 2, axis=0), 2, axis=1)
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
    panels = [(crop, "Analysed crop (224 x 224)", None), (residual, "Noise residual", "gray"),
              (spectrum, "Frequency spectrum of residual", "magma"), (np.abs(npr).mean(2)[:64, :64], "NPR residual (64 x 64 zoom)", "gray")]
    for ax, (data, title, cmap) in zip(axes, panels):
        ax.imshow(data, cmap=cmap, interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    return fig


@torch.no_grad()
def analyse(path):
    if path is None:
        return None, None, None, None, ""
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    note = f"Uploaded image: {img.size[0]} x {img.size[1]} pixels."
    if min(img.size) < SIZE:
        img = img.resize((max(SIZE, round(img.size[0] * SIZE / min(img.size))), max(SIZE, round(img.size[1] * SIZE / min(img.size)))), Image.BICUBIC)
        note += " It was smaller than 224 pixels and was enlarged, which makes the result less reliable."
    crop = np.asarray(center_crop_aligned(img))
    x = torch.from_numpy(crop.copy()).to(device).permute(2, 0, 1)[None].float() / 255
    imagenet_x = (x - IMAGENET_MEAN) / IMAGENET_STD
    clip_x = F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False) if CLIP_SIZE != SIZE else x
    emb = F.normalize(clip(pixel_values=((clip_x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float(), dim=1).cpu().numpy()

    p = {
        "M1 ResNet-50 (trained on GenImage)": torch.sigmoid(m1_old(imagenet_x)).item(),
        "M2 CLIP (trained on GenImage)": m2_old.predict_proba(emb)[0, 1],
        "M3 NPR (trained on GenImage)": torch.sigmoid(m3_old(imagenet_x)).item(),
    }
    p["M4 M2+M3 (trained on GenImage)"] = (p["M2 CLIP (trained on GenImage)"] + p["M3 NPR (trained on GenImage)"]) / 2
    p["M1 ResNet-50 (trained on OpenFake)"] = torch.sigmoid(m1_new(imagenet_x)).item()
    p["M2 CLIP (trained on OpenFake)"] = m2_new.predict_proba(emb)[0, 1]
    best = p["M2 CLIP (trained on OpenFake)"]
    verdict = {"AI-generated": float(best), "Real photograph": float(1 - best)}
    table = pd.DataFrame({
        "Detector": list(p),
        "P(AI-generated)": [f"{v:.1%}" for v in p.values()],
        "Decision": ["AI-generated" if v >= 0.5 else "Real" for v in p.values()],
    })
    attribution = torch.softmax(attributor(imagenet_x).float(), dim=1)[0].cpu().numpy()
    closest = {s: float(a) for s, a in zip(SOURCES, attribution)}
    return verdict, table, closest, fingerprint_figure(crop), note


print("analyse() ready")

## Web page

The examples are images from our WhatsApp test: each is shown as the original file and as the copy received through WhatsApp.

In [ ]:
pairs = pd.read_csv(RESULTS / "e6_whatsapp_predictions.csv")
received_dir = ROOT / "whatsapp" / "received"
examples = []
for source in ["gpt-image-2", "midjourney-7", "nano-banana-pro", "docci"]:
    row = pairs[pairs["source"] == source].iloc[0]
    examples.append([str(ROOT / "whatsapp" / "original" / row["file"])])
    if (received_dir / row["received_file"]).exists():
        examples.append([str(received_dir / row["received_file"])])

with gr.Blocks(title="AI Image Forensics") as demo:
    gr.Markdown(
        "# Is this image AI-generated?\n"
        "Upload a photo (JPEG, PNG or iPhone HEIC). The detectors look at the central 224 x 224 pixels. "
        "Research prototype from our minor project: results on new generators and on shared images are less reliable."
    )
    with gr.Row():
        with gr.Column(scale=1):
            image = gr.Image(type="filepath", label="Image")
            button = gr.Button("Analyse", variant="primary")
            note = gr.Markdown()
        with gr.Column(scale=1):
            verdict = gr.Label(label="Verdict: M2 CLIP trained on OpenFake (our best detector)")
            closest = gr.Label(label="Closest GenImage source (attribution model, E4)", num_top_classes=3)
    scores = gr.Dataframe(label="All detectors", interactive=False)
    fingerprint = gr.Plot(label="Fingerprint view")
    gr.Examples(examples=examples, inputs=image, label="Examples: original file, then the same image after WhatsApp")
    button.click(analyse, inputs=image, outputs=[verdict, scores, closest, fingerprint, note])
    image.upload(analyse, inputs=image, outputs=[verdict, scores, closest, fingerprint, note])

demo.launch(share=True, debug=True)